# Fine-tuning DenseNet-201 para Classificação de Folhas de Soja

Notebook de treinamento do modelo **DenseNet-201** com transfer learning.

**Hiperparâmetros ótimos** encontrados via Bayesian Search (Optuna):
- Dropout FC1: 0.3373
- Dropout FC2: 0.2673
- Neurônios FC1: 512
- Neurônios FC2: 256
- Ativação: ReLU
- Optimizer: Adam (lr=0.00025)
- Batch Size: 128

In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt

from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping, ModelCheckpoint
from ignite.metrics import Accuracy, Loss

from dotenv import load_dotenv
load_dotenv()

# =============================================
# CONFIGURAÇÃO
# =============================================
DATASET_PATH = os.getenv("DATASET_PATH", "/caminho/para/DADOS-DIVIDIDOS")
PRETRAINED_WEIGHTS = os.getenv("DENSENET201_PRETRAINED", "/caminho/para/models/densenet201-model-95.pth")
RESULTS_DIR = os.getenv("RESULTS_DIR", "./results/densenet201")
CHECKPOINT_DIR = os.getenv("CHECKPOINT_DIR", "./checkpoints/densenet201")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Hiperparâmetros ótimos (Bayesian Search)
BATCH_SIZE = 128
NUM_EPOCHS = 100
LEARNING_RATE = 0.00025
DROPOUT1 = 0.3372612925395379
DROPOUT2 = 0.2672714454014481
FC1_NEURONS = 512
FC2_NEURONS = 256
NUM_CLASSES = 2
FEATURE_EXTRACT = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

## 1. Data Augmentation e Carregamento do Dataset

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {
    x: datasets.ImageFolder(os.path.join(DATASET_PATH, x), data_transforms[x])
    for x in ['train', 'val', 'test']
}

dataloaders = {
    x: torch.utils.data.DataLoader(image_datasets[x], batch_size=BATCH_SIZE, shuffle=(x == 'train'), num_workers=4)
    for x in ['train', 'val', 'test']
}

class_names = image_datasets['train'].classes
print(f"Classes: {class_names}")
for split in ['train', 'val', 'test']:
    print(f"  {split}: {len(image_datasets[split])} imagens")

## 2. Carregamento do Modelo e Transfer Learning

In [ ]:
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

model = models.densenet201(pretrained=False)
set_parameter_requires_grad(model, FEATURE_EXTRACT)

# Carregar pesos pré-treinados
state_dict = torch.load(PRETRAINED_WEIGHTS, map_location=device)
del state_dict['classifier.weight']
del state_dict['classifier.bias']
model.load_state_dict(state_dict, strict=False)

# Classificador customizado
num_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT1),
    nn.Linear(num_features, FC1_NEURONS),
    nn.ReLU(),
    nn.Dropout(p=DROPOUT2),
    nn.Linear(FC1_NEURONS, FC2_NEURONS),
    nn.ReLU(),
    nn.Linear(FC2_NEURONS, NUM_CLASSES),
)

model = model.to(device)
print(f"Classificador customizado:\n{model.classifier}")

## 3. Configuração do Treinamento

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

def train_step(engine, batch):
    model.train()
    inputs, labels = batch[0].to(device), batch[1].to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    return loss.item(), outputs, labels

def eval_step(engine, batch):
    model.eval()
    with torch.no_grad():
        inputs, labels = batch[0].to(device), batch[1].to(device)
        outputs = model(inputs)
        return outputs, labels

trainer = Engine(train_step)
evaluator = Engine(eval_step)
Accuracy().attach(evaluator, 'accuracy')
Loss(criterion).attach(evaluator, 'loss')
history = {'val_loss': [], 'val_accuracy': []}

## 4. Callbacks

In [ ]:
def score_function(engine):
    return engine.state.metrics['accuracy']

early_stopping = EarlyStopping(patience=10, score_function=score_function, trainer=trainer)
evaluator.add_event_handler(Events.COMPLETED, early_stopping)

checkpoint = ModelCheckpoint(
    CHECKPOINT_DIR, filename_prefix='best', n_saved=1,
    score_function=score_function, score_name='accuracy', require_empty=False
)
evaluator.add_event_handler(Events.COMPLETED, checkpoint, {'model': model})

@trainer.on(Events.EPOCH_COMPLETED)
def log_training_results(engine):
    evaluator.run(dataloaders['val'])
    metrics = evaluator.state.metrics
    history['val_loss'].append(metrics['loss'])
    history['val_accuracy'].append(metrics['accuracy'])
    scheduler.step(metrics['loss'])
    print(f"Época {engine.state.epoch}/{NUM_EPOCHS} — Val Loss: {metrics['loss']:.4f} | Val Accuracy: {metrics['accuracy']:.4f}")

## 5. Treinamento

In [ ]:
print(f"Iniciando treinamento DenseNet-201: {NUM_EPOCHS} épocas, batch_size={BATCH_SIZE}")
trainer.run(dataloaders['train'], max_epochs=NUM_EPOCHS)
print("Treinamento finalizado!")

## 6. Avaliação no Conjunto de Teste

In [ ]:
best_model_path = os.path.join(CHECKPOINT_DIR, os.listdir(CHECKPOINT_DIR)[-1])
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

evaluator.run(dataloaders['test'])
test_metrics = evaluator.state.metrics
print(f"\nRESULTADOS NO TESTE — Accuracy: {test_metrics['accuracy']:.4f} | Loss: {test_metrics['loss']:.4f}")

## 7. Curvas de Aprendizagem

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['val_loss'], color='#e74c3c')
axes[0].set_title('Loss por Época'); axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss'); axes[0].grid(True, alpha=0.3)
axes[1].plot(history['val_accuracy'], color='#2ecc71')
axes[1].set_title('Accuracy por Época'); axes[1].set_xlabel('Época'); axes[1].set_ylabel('Accuracy'); axes[1].grid(True, alpha=0.3)
plt.suptitle('DenseNet-201 — Curvas de Aprendizagem', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'densenet201_learning_curves.png'), dpi=150, bbox_inches='tight')
plt.show()